<a href="https://colab.research.google.com/github/benjaminsuarezb-bit/L-gica-proposicional/blob/main/EA1_Ingesti%C3%B3n_de_Datos_desde_un_API_(2)_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**<span style="color:#809bd8">Curso:</span>** **Introducción a Ingeniería de Sistemas**

**<span style="color:#809bd8">Tema:</span>** **Evidencia de Aprendizaje EA1**

**<span style="color:#809bd8">Estudiante:</span>** **Benjamín Suárez Báez**

**<span style="color:#809bd8">Profesor:</span>** **Walter Hugo Arboleda Mazo**

**<span style="color:#809bd8">Universidad:</span>** **UNAC**

**<span style="color:#809bd8">Fecha:</span>** **21/02/2026**

In [ ]:
# pip install requests pandas openpyxl

In [ ]:
import requests # importacion de la libreria Requests para enviar peticion a la API https://jsonplaceholder.typicode.com/users
import sqlite3 # importacion de la interface para conexion y creacion de bases de datos SQLite
import pandas as pd # importacion de Pandas, libreria para el analisis de datos en Python

# Definicion de la URL de la API para envio de peticion y recepcion de datos usando protocolo HTTP
# desde la API https://jsonplaceholder.typicode.com/
# Una API se puede definir como el puente que conecta dos aplicacines facilitando que una de ellas pueda aprovechar las funciones o servicios que la otra ofrece
# JSON es basicamente un formato basado en texto que es legible para los humanos y que cumple la función de poder intercambiar y almacenar datos estructurados
API_URL = "https://jsonplaceholder.typicode.com/users"


# Creacion de la funcion extraer_datos validandose codigo 200 (exito en la conexion) o codigo 404 (codigo de error en la conexion)
# Para que el proceso funcione es necesario enviar una solicitud a un servidor para poder obtener la información requerida
"""
   Se puede observar un código de respuesta en la condición. Estos códigos sirven para indicar el resultado de una solicitud y se clasifican en:
   * 1xx (Información): La solicitud ha sido recibida y el proceso continúa.
   * 2xx (Éxito): La solicitud se procesó correctamente.
   * 3xx (Redirección): Se necesitan acciones adicionales para completar la solicitud.
   * 4xx (Error del cliente): El cliente ha hecho una solicitud incorrecta.
   * 5xx (Error del servidor): El servidor no pudo completar la solicitud.
"""
def extraer_datos(url):
    response = requests.get(url) # Aquí se puede observar que se envía una petición (GET)
    if response.status_code == 200: # si todo sole bien se devuelven los datos en formato JSON
        return response.json()
    else:
        raise Exception(f"Error al conectar con la API: {response.status_code}")


# Creacion de la base de datos en SQLite "datos_api.db"
# se realiza la conexion a dicha base de datos
# y se crea el "cursor" para "execute" el CREATE TABLE y el INSERT OR REPLACE INTO usuarios
"""
   Una base de datos es una recopilación organizada de información o datos estructurados,
   almacenada electrónicamente en un sistema informático, que permite gestionar, actualizar,
   acceder y analizar información rápidamente
"""
def guardar_en_db(datos): # Esta función se conecta a la base de datos datos_api.db
    conn = sqlite3.connect('datos_api.db')
    cursor = conn.cursor()

    # Creación de la tabla "usuarios" dentro de la base de datos "datos_api.db"
    # esta contiene los campos id, nombre, usuario y email
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS usuarios (
            id INTEGER PRIMARY KEY,
            nombre TEXT,
            usuario TEXT,
            email TEXT
        )
    ''')

    # Inserción de datos en la tabla "usuarios"
    # INSERT OR REPLACE es un comando de bases de datos (SQL) significa que si el id o registro ya existe, lo reemplaza, y si no existe, lo inserta
    for item in datos:
        cursor.execute('''
            INSERT OR REPLACE INTO usuarios (id, nombre, usuario, email)
            VALUES (?, ?, ?, ?)
        ''', (item['id'], item['name'], item['username'], item['email']))

    # Aceptación mediante "Commit" el ingreso de datos en la tabla "usuarios"
    conn.commit() # Con el commit() se confirman los cambios

    # Cerrado de la conexion a la base de datos "datos_api.db"
    conn.close() # Con el close() se cierra la conexión

# 3. GENERAR EVIDENCIAS (Pandas y Auditoría)
def generar_evidencias(datos_api): # Esta función valida que los datos se hayan guardado correctamente

    # Un DataFrame es una estructura de datos organizada en filas y columnas, similar a Excel
    # Vale la pena mencionar que Pandas es una biblioteca de Python diseñada para la manipulación y análisis de datos
    # Pandas contiene estructuras de datos muy eficientes como los DataFrame
    # Un archivo CSV es un tipo de archivo de texto plano que almacena datos estructurados en forma de tabla (con filas y columnas)
    """
    En la siguiente sección pasa lo siguiente:
    *Se leen los datos desde la base de datos SQL
    *Se convierten los datos en un DataFrame de Pandas
    *Se crea un archivo Excel/CSV (muestra_usuarios.csv) con los datos del DataFrame
    """
      # --- Archivo Excel/CSV con Pandas ---
    conn = sqlite3.connect('datos_api.db')
    df_db = pd.read_sql_query("SELECT * FROM usuarios", conn)
    df_db.to_csv('muestra_usuarios.csv', index=False)
    conn.close()

    # la variable "registros_api" obtiene la cantidad de registros recibidos desde la
    # API con url "https://jsonplaceholder.typicode.com/users"
    # la variable "registros_db" obtiene la cantidad de registros almacenados en la tabla "usuarios"
    """
    Hasta la línea 108 se compara cuántos registros vinieron desde la API y cuántos registros quedaron en la base,
    se crea un archivo (auditoria.txt) con esos datos, y si los números de ambas variables coinciden la operación es exitosa,
    de lo contrario es fallida

    """
    registros_api = len(datos_api)
    registros_db = len(df_db)

    # Creación del archivo "auditoría.txt" con el metodo "open" de Python
    with open('auditoria.txt', 'w', encoding='utf-8') as f:
        f.write("RESUMEN DE AUDITORÍA\n")
        f.write("====================\n")
        f.write(f"Registros extraídos de la API https://jsonplaceholder.typicode.com/: {registros_api}\n")
        f.write(f"Registros guardados en la DB de SQLite3: {registros_db}\n")

        if registros_api == registros_db:
            f.write("\nESTADO: Operacion Exitosa.\n")
        else:
            f.write("\nESTADO: Operacion Fallida.\n")

# EJECUCIÓN DEL PROCESO
# Este "try" y "except" nos ayudan a ejecutar todo el código en un bloque de contro de errores
try:
    print("Iniciando proceso de petición y recepción de datos desde la API https://jsonplaceholder.typicode.com/users...")
    datos = extraer_datos(API_URL) # Se extraen los datos

    print("Creando la base de datos datos_api.db y guardando datos en la tabla usuarios...")
    guardar_en_db(datos) # Los datos se guardan en SQLite

    print("Generando archivos de evidencia .CSV y auditoría.txt...")
    generar_evidencias(datos) # Se genera el CSV y el archivo auditoría

    print("¡Proceso completado con éxito!")
except Exception as e: # Si ocurre un error lo muestra en pantalla
    print(f"Ocurrió un error: {e}")

Iniciando proceso de petición y recepción de datos desde la API https://jsonplaceholder.typicode.com/users...
Creando la base de datos datos_api.db y guardando datos en la tabla usuarios...
Generando archivos de evidencia .CSV y auditoría.txt...
¡Proceso completado con éxito!



**Referencias**

https://realpython.com/python-requests/#make-a-get-request

https://pandas.pydata.org/

https://pypi.org/project/openpyxl/

https://docs.python.org/3/library/sqlite3.html

https://sqliteviewer.app/

https://sibabalwesinyaniso.medium.com/what-is-a-cursor-understanding-how-sqlite-handles-queries-in-python-8f8c88546820
https://www.pythonlore.com/understanding-sqlite3-cursor-object-for-database-operations/
https://miro.medium.com/v2/resize:fit:720/format:webp/0*rePKZ6jMZbZ_6S_3.gif
